# 次季营收指引：来源可行性

只检查一个潜在新信息：NVDA 首次财报中的次季营收指引相对已报告当季营收。公开经营前瞻与历史 vendor EPS consensus surprise 不同；尚未预登记模型，也不声称超额。

本 notebook 只资格审查 SEC 官方申报元数据，用 Qlib 原生交易日历确定保守的潜在可用日，将完整目录留在 Recorder。正文、数值与完整披露事件仍待核验。不能据此生成 feature store 或训练。

In [1]:
from pathlib import Path
import sys,os,json,hashlib
import numpy as np,pandas as pd,yaml
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE','true')
ROOT=Path.cwd()
if not (ROOT/'src').is_dir(): ROOT=ROOT.parents[1]
sys.path.insert(0,str(ROOT))
CARD=ROOT/'research/nvda_revenue_guidance_source_intake_20261011'
RAW=ROOT/'data/raw/external/sec_nvda_submissions_20261011_v1'
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.provider_content_hash import provider_content_hash
STORE=ROOT/'data/qlib/published_futu_affine_20260925_v1'
assert provider_content_hash(STORE)[0]=='b46e8edf02f46f15a75e07ded378e83bd13b6305fa98329754b23ea4cfec0efa'
qlib.init(provider_uri=str(STORE),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(freq='day'))
assert str(calendar[-1].date())=='2026-09-25'
manifest=json.loads((RAW/'manifest.json').read_text())
frames=[]
for entry in manifest['entries']:
    path=ROOT/entry['path'];payload=path.read_bytes()
    assert hashlib.sha256(payload).hexdigest()==entry['sha256']
    data=json.loads(payload)
    rows=data['filings']['recent'] if 'filings' in data else data
    frame=pd.DataFrame(rows);frame['catalog_url']=entry['url'];frames.append(frame)
all_filings=pd.concat(frames,ignore_index=True)
assert all_filings.accessionNumber.is_unique
assert all_filings.acceptanceDateTime.map(lambda x:pd.Timestamp(x).tzinfo is not None).all()
all_filings['accepted_at']=pd.to_datetime(all_filings.acceptanceDateTime,utc=True)
cutoff=pd.Timestamp('2026-09-25 16:15',tz='America/New_York')
mask=all_filings.form.isin(['8-K','8-K/A']) & all_filings['items'].str.contains('2.02',regex=False) & all_filings.filingDate.ge('2011-01-01')
inventory=all_filings.loc[mask & all_filings.accepted_at.le(cutoff)].sort_values('accepted_at').copy()
assert inventory['items'].ne('').all() and not inventory.empty
ny_days=inventory.accepted_at.dt.tz_convert('America/New_York').dt.tz_localize(None).dt.normalize()
date_bound=pd.concat([ny_days,pd.to_datetime(inventory.filingDate),pd.to_datetime(inventory.reportDate)],axis=1).max(axis=1)
positions=calendar.searchsorted(pd.DatetimeIndex(date_bound),side='right')
assert (positions<len(calendar)).all()
inventory['potential_safe_signal_day']=calendar[positions].to_numpy()
clock=(pd.DatetimeIndex(inventory.potential_safe_signal_day).tz_localize('America/New_York')+pd.Timedelta(hours=16,minutes=15)).tz_convert('UTC')
assert (clock>inventory.accepted_at.to_numpy()).all()
inventory['guidance_value_known']=False
assert not inventory.guidance_value_known.any()
years=inventory.groupby(inventory.filingDate.str[:4]).size().rename('item202_filing_count').reset_index().rename(columns={'filingDate':'year'})
print('Official filing inventory:',len(inventory),'all catalogs:',len(all_filings),'post-cutoff catalog rows:',int((all_filings.accepted_at>cutoff).sum()))
print(years.to_string(index=False))

web_manifest=json.loads((RAW/'web_extract_manifest.json').read_text())
web_bytes=(RAW/'official_web_index_facts.json').read_bytes()
assert hashlib.sha256(web_bytes).hexdigest()==web_manifest['snapshot_sha256']
web_facts=json.loads(web_bytes)
clock_rows=[]
for fact in web_facts['facts']:
    projection={k:v for k,v in fact.items() if k not in web_manifest['normalized_fact_sha256_excluded_fields']}
    assert hashlib.sha256(json.dumps(projection,sort_keys=True,separators=(',',':')).encode()).hexdigest()==fact['normalized_fact_sha256']
    assert fact['original_html_sha256'] is None and fact['original_html_bytes_available'] is False
    row=inventory.loc[inventory.accessionNumber.eq(fact['accessionNumber'])]
    assert len(row)==1 and row.iloc[0].acceptanceDateTime==fact['api_accepted_text']
    api=pd.Timestamp(fact['api_accepted_text'])
    assumed=pd.Timestamp(fact['index_accepted_text'],tz='America/New_York').tz_convert('UTC')
    clock_rows.append(dict(accession=fact['accessionNumber'],api_accepted_raw=fact['api_accepted_text'],
        index_accepted_raw=fact['index_accepted_text'],index_timezone_assumption='America/New_York_unverified',
        api_minus_assumed_index_hours=(api-assumed).total_seconds()/3600,
        clock_conflict=api!=assumed,first_public_timestamp_qualified=False))
clock_audit=pd.DataFrame(clock_rows)
assert len(clock_audit)==7 and clock_audit.clock_conflict.all()
clock_audit.to_csv(CARD/'source_clock_audit.csv',index=False)
print('Official index anchors7; API-vs-index clock conflicts7. No guessed timezone correction.')


Official filing inventory: 66 all catalogs: 2488 post-cutoff catalog rows: 1
year  item202_filing_count
2011                     4
2012                     4
2013                     4
2014                     5
2015                     4
2016                     4
2017                     4
2018                     4
2019                     5
2020                     4
2021                     4
2022                     5
2023                     4
2024                     4
2025                     4
2026                     3
Official index anchors7; API-vs-index clock conflicts7. No guessed timezone correction.


In [2]:
experiment=CARD.name;uri=str(CARD/'qlib_recorder')
if (CARD/'summary.json').exists():
    from src.core.recorder_sources import load_verified_recorder_objects
    saved=json.loads((CARD/'summary.json').read_text())
    objects,audit=load_verified_recorder_objects(CARD,expected_recorder_id=saved['recorder_id'],
        artifacts=['intake/earnings_item_inventory.pkl','intake/source_clock_audit.pkl','intake/official_index_extract.pkl'])
    pd.testing.assert_frame_equal(objects['intake/earnings_item_inventory.pkl'],inventory,check_exact=True)
    pd.testing.assert_frame_equal(objects['intake/source_clock_audit.pkl'],clock_audit,check_exact=True)
    assert objects['intake/official_index_extract.pkl']==web_facts
    rid=saved['recorder_id'];local=saved['recorder_local_dir']
else:
    with R.start(experiment_name=experiment,recorder_name='metadata_intake_only',uri=uri):
        rec=R.get_recorder()
        rec.log_params(source='SEC submissions API',stage='metadata_only',fit_calls=0,replays=0,market_cutoff='2026-09-25')
        rec.save_objects(artifact_path='intake',**{'all_filing_metadata.pkl':all_filings,'earnings_item_inventory.pkl':inventory,'source_manifest.pkl':manifest,'native_calendar.pkl':calendar,'official_index_extract.pkl':web_facts,'source_clock_audit.pkl':clock_audit})
        rid=rec.id;local=rec.get_local_dir()
    assert rec.status=='FINISHED'
policy=dict(policy='slim_card_recorder_first',workflow_config='workflow.yaml',recorder_only_artifacts=['intake/all_filing_metadata.pkl','intake/earnings_item_inventory.pkl','intake/source_manifest.pkl','intake/native_calendar.pkl','intake/official_index_extract.pkl','intake/source_clock_audit.pkl'])
summary=dict(experiment=experiment,status='needs_data_guidance_body_and_full_event_qualification',source_deployment_gate=False,source_qualified=False,promotion_decision='not_promoted',promotion_eligible=False,fit_calls=0,native_cases=0,filing_inventory_rows=len(inventory),source_feature_values_known=0,official_index_anchors=7,clock_conflict_anchors=7,api_timestamp_semantics_qualified=False,all_catalog_rows=len(all_filings),excluded_post_cutoff_metadata_rows=int((all_filings.accepted_at>cutoff).sum()),source=dict(provider_uri=str(STORE.relative_to(ROOT)),region='us',as_of='2026-09-25',raw_manifest=str((RAW/'manifest.json').relative_to(ROOT))),recorder_id=rid,recorder_uri=uri,recorder_local_dir=local,recorder_status='FINISHED',artifact_policy=policy,limitations=['SEC current API is metadata only; no revenue guidance values','Item2.02 filings are not automatically quarterly earnings or forward guidance; interim updates retained','Direct SEC original-index request returned403; web-tool official document extraction remains a possible separately labelled path','Current issuer HTML not proven byte-identical to as-filed exhibits','No final model hypothesis, budgets or financial gates registered','API declared UTC strings conflict with index local times; preserve both, no guessed correction','Official web index extracts7 of66; some exact filing indexes unavailable via web tool too'])
(CARD/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n')
(CARD/'workflow.yaml').write_text(yaml.safe_dump(dict(experiment=experiment,source=summary['source'],recorder_uri=uri,recorder_id=rid,artifact_policy=policy),sort_keys=False))
(CARD/'selected_model.json').write_text(json.dumps(dict(model=None,status='source_intake_only',promotion_decision='not_promoted'),indent=2)+'\n')
years.to_csv(CARD/'filing_coverage.csv',index=False)
entries=[]
for p in sorted((Path(local)/'artifacts').rglob('*')):
    if p.is_file() and not p.name.startswith('code_'):entries.append(dict(path=str(p.relative_to(CARD)),sha256=hashlib.sha256(p.read_bytes()).hexdigest(),bytes=p.stat().st_size))
(CARD/'verification_manifest.json').write_text(json.dumps(dict(recorder_id=rid,entries=entries),indent=2)+'\n')
conclusion=f"""# 次季营收指引来源：元数据接入

已核验 SEC 两份官方目录，共 {len(all_filings)} 个 accession，无重复。2011至截止日前的 Item2.02 申报 {len(inventory)} 个；完整目录与 Qlib 原生日历已归档。包含截止日后的目录条目明确排除，不作为财务特征；价格日历未缩短。

经营前瞻与已失败 vendor EPS surprise 不同，但这里只获得申报元数据：指引已知值0，0次拟合、0次回放。Item2.02不自动等于季度指引，包含中期更新；原件正文、目标财季、数值口径、更新/撤回、完整覆盖均未通过。

官方web索引事实已取得7条，但至少2个原件索引也不可读取，全66条正文和指引仍未取得。提取事实hash不是原始HTML hash。2014-05-06、2019-01-28、2022-08-08三条额外事件保留，未冒充季度指引。

7条索引与API时钟有冲突：2011 API声明2011-02-17T02:22:17Z，而索引显示2011-02-16 16:22:17；不猜时区修正，不认证first-public。潜在可用日仅以较晚元数据日期后的Qlib session作为保守边界，尚不能作为已知特征。

状态 needs data，不晋升。下一步：完整取得并核验官方备案正文/指引事件；通过后才冻结单一信息假设和有限实验。不能回到已关闭H20或VXN/VIX路线调参。

Recorder：{rid}；FINISHED。
"""
(CARD/'conclusion.md').write_text(conclusion)
print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "experiment": "nvda_revenue_guidance_source_intake_20261011",
  "status": "needs_data_guidance_body_and_full_event_qualification",
  "source_deployment_gate": false,
  "source_qualified": false,
  "promotion_decision": "not_promoted",
  "promotion_eligible": false,
  "fit_calls": 0,
  "native_cases": 0,
  "filing_inventory_rows": 66,
  "source_feature_values_known": 0,
  "official_index_anchors": 7,
  "clock_conflict_anchors": 7,
  "api_timestamp_semantics_qualified": false,
  "all_catalog_rows": 2488,
  "excluded_post_cutoff_metadata_rows": 1,
  "source": {
    "provider_uri": "data/qlib/published_futu_affine_20260925_v1",
    "region": "us",
    "as_of": "2026-09-25",
    "raw_manifest": "data/raw/external/sec_nvda_submissions_20261011_v1/manifest.json"
  },
  "recorder_id": "917e5a134fc34422b62c63972d1cc75e",
  "recorder_uri": "/Users/rziiiii/dev/workspace/research/nvda_revenue_guidance_source_intake_20261011/qlib_recorder",
  "recorder_local_dir": "/Users/rziiiii/dev/workspace/

# 次季营收指引来源：元数据接入

已核验 SEC 两份官方目录，共 2488 个 accession，无重复。2011至截止日前的 Item2.02 申报 66 个；完整目录与 Qlib 原生日历已归档。包含截止日后的目录条目明确排除，不作为财务特征；价格日历未缩短。

经营前瞻与已失败 vendor EPS surprise 不同，但这里只获得申报元数据：指引已知值0，0次拟合、0次回放。Item2.02不自动等于季度指引，包含中期更新；原件正文、目标财季、数值口径、更新/撤回、完整覆盖均未通过。

官方web索引事实已取得7条，但至少2个原件索引也不可读取，全66条正文和指引仍未取得。提取事实hash不是原始HTML hash。2014-05-06、2019-01-28、2022-08-08三条额外事件保留，未冒充季度指引。

7条索引与API时钟有冲突：2011 API声明2011-02-17T02:22:17Z，而索引显示2011-02-16 16:22:17；不猜时区修正，不认证first-public。潜在可用日仅以较晚元数据日期后的Qlib session作为保守边界，尚不能作为已知特征。

状态 needs data，不晋升。下一步：完整取得并核验官方备案正文/指引事件；通过后才冻结单一信息假设和有限实验。不能回到已关闭H20或VXN/VIX路线调参。

Recorder：917e5a134fc34422b62c63972d1cc75e；FINISHED。
